# Setting up

In [10]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import glob

# Load phenotype clusters

In [20]:
phenotype_clusters = pd.read_csv("/gpfs/home/yb2612/proteomics/data/Cervical_mIF/output/initial_run/clustering/27_clusters/phenotype_clusters.csv", index_col=0)
print(phenotype_clusters.shape)
# Extract the third chunk split by "-" (e.g., "02433" from "20250225-Jharna-02433-A1_Scan1.er")
phenotype_clusters["short_id"] = phenotype_clusters["slide_id"].apply(lambda x: x.split("-")[2])
phenotype_clusters["short_id"].value_counts()
# remove all rows where short id is 07611
phenotype_clusters = phenotype_clusters[phenotype_clusters["short_id"] != "07611"]
phenotype_clusters

(7783475, 2)


,slide_id,cluster_label,short_id
1,20250225-Jharna-02433-A1_Scan1.er,17,02433
2,20250225-Jharna-02433-A1_Scan1.er,1,02433
3,20250225-Jharna-02433-A1_Scan1.er,17,02433
4,20250225-Jharna-02433-A1_Scan1.er,17,02433
5,20250225-Jharna-02433-A1_Scan1.er,8,02433
...,...,...,...
7783471,20250318-Jharna-28873-A1_Scan1.er,7,28873
7783472,20250318-Jharna-28873-A1_Scan1.er,26,28873
7783473,20250318-Jharna-28873-A1_Scan1.er,17,28873
7783474,20250318-Jharna-28873-A1_Scan1.er,26,28873


# Detailed cell types

In [22]:
# dir containing celesta result folders
celesta_results_dir = "/gpfs/home/yb2612/yb2612_fenyo/results/celesta/detailed_cell_types"
output_dir = "/gpfs/home/yb2612/proteomics/data/Cervical_mIF/output/initial_run/celltypes/27_clusters"

thresholds_dict = {
    "high_anchor_default_high_iter_default":"default",
    "high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.8_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.6_0.5_0.5_0.5_0.5_0.5_0.5_0.5":"B_test_high-anchor_0.8_high_iter_0.6",
    "high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.8_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.7_0.5_0.5_0.5_0.5_0.5_0.5_0.5":"B_test_high-anchor_0.8_high_iter_0.7",
    "high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.8_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_default":"B_test_high-anchor_0.8_high_iter_0.5",
    "high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.9_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.6_0.5_0.5_0.5_0.5_0.5_0.5_0.5":"B_test_high-anchor_0.9_high_iter_0.6",
    "high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.9_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.7_0.5_0.5_0.5_0.5_0.5_0.5_0.5":"B_test_high-anchor_0.9_high_iter_0.7",
    "high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.9_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_0.5_0.5_0.5_0.5_0.5_0.5_0.8_0.5_0.5_0.5_0.5_0.5_0.5_0.5":"B_test_high-anchor_0.9_high_iter_0.8",
    "high_anchor_0.7_0.7_0.7_0.7_0.7_0.7_0.9_0.7_0.7_0.7_0.7_0.7_0.7_0.7_high_iter_default":"B_test_high-anchor_0.9_high_iter_0.5"
}

# each folder here is named cervical_{short_id}_raw_arcsinh
# extract short_id and find file ending with {threshold}_final_cell_type_assignment.csv
# take "Final cell type" col and add it to a new col in phenotype_clusters named cell_type for that short_id
# repeat for all samples
# make sure order of rows is preserved per sample

for threshold, table_name in thresholds_dict.items():

    pc_copy = phenotype_clusters.copy()  # create copy of phenotype_clusters
    # Initialize a column for cell types
    pc_copy["cell_type"] = None
    
    # Process each sample
    for short_id in pc_copy["short_id"].unique():
        print(short_id)
        folder_name = f"cervical_{short_id}_raw_arcsinh"
        folder_path = os.path.join(celesta_results_dir, folder_name)
    
        if not os.path.isdir(folder_path):
            print(f"Folder not found for short_id: {short_id}")
            continue
    
        # Find the final assignment file
        file_pattern = os.path.join(folder_path, f"*{threshold}_final_cell_type_assignment.csv")
        matches = glob.glob(file_pattern)
    
        if len(matches) != 1:
            print(f"Expected 1 file, found {len(matches)} for {short_id}")
            continue
    
        # Read final cell type assignments
        celesta_df = pd.read_csv(matches[0])
        final_cell_types = celesta_df["Final cell type"].values
    
        # Get the phenotype rows for this short_id
        mask = pc_copy["short_id"] == short_id
        n_rows = mask.sum()
    
        if len(final_cell_types) != n_rows:
            print(f"Row mismatch for {short_id}: phenotype has {n_rows}, celesta has {len(final_cell_types)}")
            continue
    
        # Assign cell types, preserving order
        pc_copy.loc[mask, "cell_type"] = final_cell_types
    
    # save
    print("saving")
    output_path = os.path.join(output_dir, f"celesta_{table_name}.csv")
    pc_copy.to_csv(output_path, index=False)
    print("saved")

02433
08153
10103
00862
09002
10285
34933
00438
07688
39367
49411
04738
07291
28873
saving
saved
02433
08153
Expected 1 file, found 0 for 08153
10103
00862
Expected 1 file, found 0 for 00862
09002
Expected 1 file, found 0 for 09002
10285
Expected 1 file, found 0 for 10285
34933
00438
Expected 1 file, found 0 for 00438
07688
Expected 1 file, found 0 for 07688
39367
49411
04738
Expected 1 file, found 0 for 04738
07291
Expected 1 file, found 0 for 07291
28873
saving
saved
02433
08153
Expected 1 file, found 0 for 08153
10103
00862
Expected 1 file, found 0 for 00862
09002
Expected 1 file, found 0 for 09002
10285
Expected 1 file, found 0 for 10285
34933
00438
Expected 1 file, found 0 for 00438
07688
Expected 1 file, found 0 for 07688
39367
49411
04738
Expected 1 file, found 0 for 04738
07291
Expected 1 file, found 0 for 07291
28873
saving
saved
02433
08153
Expected 1 file, found 0 for 08153
10103
00862
Expected 1 file, found 0 for 00862
09002
Expected 1 file, found 0 for 09002
10285
Expected

## Fine cell types

In [18]:
detailed_cell_types = pd.read_csv("/gpfs/home/yb2612/proteomics/data/Cervical_mIF/output/initial_run/celltypes/27_clusters/cell_types_detailed_default.csv")
detailed_cell_types

,slide_id,cluster_label,short_id,cell_type
0,20250225-Jharna-02433-A1_Scan1.er,17,2433,Tumor
1,20250225-Jharna-02433-A1_Scan1.er,1,2433,Tumor
2,20250225-Jharna-02433-A1_Scan1.er,17,2433,Tumor
3,20250225-Jharna-02433-A1_Scan1.er,17,2433,Tumor
4,20250225-Jharna-02433-A1_Scan1.er,8,2433,Th1
...,...,...,...,...
6415274,20250318-Jharna-28873-A1_Scan1.er,7,28873,Stromal_Undefined
6415275,20250318-Jharna-28873-A1_Scan1.er,26,28873,Tumor
6415276,20250318-Jharna-28873-A1_Scan1.er,17,28873,Tumor
6415277,20250318-Jharna-28873-A1_Scan1.er,26,28873,T


In [19]:
# create dict of short_id to slide_id and look for only unique values in df
slide_id_dict = {}
for short_id in detailed_cell_types["short_id"].unique():
    slide_id = detailed_cell_types[detailed_cell_types["short_id"] == short_id]["slide_id"].values[0]
    slide_id_dict[short_id] = slide_id
print(slide_id_dict)

{np.int64(2433): '20250225-Jharna-02433-A1_Scan1.er', np.int64(8153): '20250225-Jharna-08153-A1_Scan1.er', np.int64(10103): '20250225-Jharna-10103-A1_Scan1.er', np.int64(862): '20250305-Jharna-00862-6X_Scan1.er', np.int64(9002): '20250305-Jharna-09002-A1_Scan1.er', np.int64(10285): '20250305-Jharna-10285-A21_Scan1.er', np.int64(34933): '20250305-Jharna-34933-A1_Scan1.er', np.int64(438): '20250311-Jharna-00438-1A_Scan1.er', np.int64(7688): '20250311-Jharna-07688-A1_Scan1.er', np.int64(39367): '20250311-Jharna-39367-A1_Scan1.er', np.int64(49411): '20250311-Jharna-49411-1A_Scan1.er', np.int64(4738): '20250318-Jharna-04738-A1_Scan1.er', np.int64(7291): '20250318-Jharna-07291-1A_Scan1.er', np.int64(28873): '20250318-Jharna-28873-A1_Scan1.er'}


In [20]:
# pad short id to 5 digits with leading zeros
detailed_cell_types["short_id"] = detailed_cell_types["short_id"].astype(str).str.zfill(5)
# add col cell_id to detailed_cell_types by doing short_id_Cell<no of cell in that short_id>
detailed_cell_types["cell_id"] = (
    detailed_cell_types
    .groupby("short_id")
    .cumcount()
    .add(1)
    .astype(str)
    .radd(detailed_cell_types["short_id"] + "_Cell")
)
detailed_cell_types

,slide_id,cluster_label,short_id,cell_type,cell_id
0,20250225-Jharna-02433-A1_Scan1.er,17,02433,Tumor,02433_Cell1
1,20250225-Jharna-02433-A1_Scan1.er,1,02433,Tumor,02433_Cell2
2,20250225-Jharna-02433-A1_Scan1.er,17,02433,Tumor,02433_Cell3
3,20250225-Jharna-02433-A1_Scan1.er,17,02433,Tumor,02433_Cell4
4,20250225-Jharna-02433-A1_Scan1.er,8,02433,Th1,02433_Cell5
...,...,...,...,...,...
6415274,20250318-Jharna-28873-A1_Scan1.er,7,28873,Stromal_Undefined,28873_Cell26050
6415275,20250318-Jharna-28873-A1_Scan1.er,26,28873,Tumor,28873_Cell26051
6415276,20250318-Jharna-28873-A1_Scan1.er,17,28873,Tumor,28873_Cell26052
6415277,20250318-Jharna-28873-A1_Scan1.er,26,28873,T,28873_Cell26053


In [7]:
metadata = pd.read_csv('/gpfs/home/yb2612/yb2612_fenyo/data/seurat_objects/Cervical_v5_obj_metadata_total_metadata_radial_distances_kmeans_fixed_parent_tiles_updated.csv')

/tmp/ipykernel_2892471/2605052530.py:1: DtypeWarning: Columns (20,27) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv('/gpfs/home/yb2612/yb2612_fenyo/data/seurat_objects/Cervical_v5_obj_metadata_total_metadata_radial_distances_kmeans_fixed_parent_tiles_updated.csv')


In [16]:
metadata[["X","barcodes"]]

,X,barcodes
0,02433_Cell104196,Cell104196
1,02433_Cell34765,Cell34765
2,02433_Cell49960,Cell49960
3,02433_Cell22080,Cell22080
4,02433_Cell85101,Cell85101
...,...,...
6415274,28873_Cell5306,Cell5306
6415275,28873_Cell5326,Cell5326
6415276,28873_Cell5298,Cell5298
6415277,28873_Cell5266,Cell5266


In [8]:
metadata.head()

,Unnamed: 0,X,orig.ident,nCount_RNA,nFeature_RNA,Final.cell.type,Fine.cell.type,Fine.cell.type2,Parent.cell.type,Grandparent.cell.type,...,Activated_Treg_dist_kmeans,Macrophage_CD163neg_dist_kmeans,Activated_Th1_dist_kmeans,Activated_Macrophage_CD163neg_dist_kmeans,Activated_Macrophage_CD163pos_dist_kmeans,HPV,Race,Ethnicity,Language,Smoking
0,1,02433_Cell104196,2433,12822.125000,34,Macrophage_CD163pos,Macrophage_CD163pos,Macrophage_CD163pos,Macrophage,Macrophage,...,Moderately_near,Moderately_near,Moderately_near,Near,Near,Positive,Other,Hispanic,Spanish,Yes
1,2,02433_Cell34765,2433,10075.666669,34,Th1,Th1,Th1,CD4_T,T,...,Far,Near,Moderately_near,Moderately_near,Near,Positive,Other,Hispanic,Spanish,Yes
2,3,02433_Cell49960,2433,2882.000049,29,Stromal_Undefined,Stromal_Undefined_Unknown,Stromal_Undefined,Stromal_Undefined_Unknown,Stromal_Undefined_Unknown,...,Far,Near,Near,Near,Near,Positive,Other,Hispanic,Spanish,Yes
3,4,02433_Cell22080,2433,4377.499977,34,Stromal_Undefined,Stromal_Undefined_Unknown,Stromal_Undefined,Stromal_Undefined_Unknown,Stromal_Undefined_Unknown,...,Far,Moderately_near,Far,Moderately_near,Moderately_near,Positive,Other,Hispanic,Spanish,Yes
4,5,02433_Cell85101,2433,17595.433251,33,Tumor,Tumor,Tumor,Tumor,Tumor,...,Far,Moderately_near,Far,Near,Near,Positive,Other,Hispanic,Spanish,Yes


In [21]:
metadata_trimmed = metadata[["X","orig.ident","Fine.cell.type"]].copy()
metadata_trimmed.columns

Index(['X', 'orig.ident', 'Fine.cell.type'], dtype='object')

In [22]:
# rename X to cell_id
metadata_trimmed.rename(columns={"X": "cell_id"}, inplace=True)
metadata_trimmed.rename(columns={"orig.ident": "short_id"}, inplace=True)
# map slide_id from slide_id_dict to short_id in metadata_trimmed
metadata_trimmed["slide_id"] = metadata_trimmed["short_id"].map(slide_id_dict)
# rearrange columns to have slide_id first
metadata_trimmed = metadata_trimmed[["slide_id", "short_id", "cell_id", "Fine.cell.type"]]
metadata_trimmed

,slide_id,short_id,cell_id,Fine.cell.type
0,20250225-Jharna-02433-A1_Scan1.er,2433,02433_Cell104196,Macrophage_CD163pos
1,20250225-Jharna-02433-A1_Scan1.er,2433,02433_Cell34765,Th1
2,20250225-Jharna-02433-A1_Scan1.er,2433,02433_Cell49960,Stromal_Undefined_Unknown
3,20250225-Jharna-02433-A1_Scan1.er,2433,02433_Cell22080,Stromal_Undefined_Unknown
4,20250225-Jharna-02433-A1_Scan1.er,2433,02433_Cell85101,Tumor
...,...,...,...,...
6415274,20250318-Jharna-28873-A1_Scan1.er,28873,28873_Cell5306,Cycling_Tumor
6415275,20250318-Jharna-28873-A1_Scan1.er,28873,28873_Cell5326,Endothelial
6415276,20250318-Jharna-28873-A1_Scan1.er,28873,28873_Cell5298,Tumor
6415277,20250318-Jharna-28873-A1_Scan1.er,28873,28873_Cell5266,Neutrophil


In [39]:
# convert cell_id to string
metadata_trimmed["cell_id"] = metadata_trimmed["cell_id"].astype(str)
detailed_cell_types["cell_id"] = detailed_cell_types["cell_id"].astype(str)

# merge
metadata_joined = detailed_cell_types[["cell_id","cell_type","cluster_label"]].merge(
    metadata_trimmed[["cell_id"] + [col for col in metadata_trimmed.columns if col != "cell_id"]],
    on="cell_id",
    how="left"
)

In [35]:
metadata_joined

,cell_id,cell_type,slide_id,short_id,Fine.cell.type
0,02433_Cell1,Tumor,20250225-Jharna-02433-A1_Scan1.er,2433,Tumor
1,02433_Cell2,Tumor,20250225-Jharna-02433-A1_Scan1.er,2433,Cycling_Tumor
2,02433_Cell3,Tumor,20250225-Jharna-02433-A1_Scan1.er,2433,Cycling_Tumor
3,02433_Cell4,Tumor,20250225-Jharna-02433-A1_Scan1.er,2433,Cycling_Tumor
4,02433_Cell5,Th1,20250225-Jharna-02433-A1_Scan1.er,2433,Th1
...,...,...,...,...,...
6415274,28873_Cell26050,Stromal_Undefined,20250318-Jharna-28873-A1_Scan1.er,28873,Stromal_Undefined_Unknown
6415275,28873_Cell26051,Tumor,20250318-Jharna-28873-A1_Scan1.er,28873,Tumor
6415276,28873_Cell26052,Tumor,20250318-Jharna-28873-A1_Scan1.er,28873,Tumor
6415277,28873_Cell26053,T,20250318-Jharna-28873-A1_Scan1.er,28873,T


In [36]:
# ensure no na
metadata_joined.isna().sum()

cell_id           0
cell_type         0
slide_id          0
short_id          0
Fine.cell.type    0
dtype: int64

In [41]:
# pad short id to 5 digits with leading zeros
metadata_joined["short_id"] = metadata_joined["short_id"].astype(str).str.zfill(5)
# drop cell_type
metadata_joined.drop(columns = "cell_type", inplace=True)
# rename Fine.cell.type to cell_type
metadata_joined.rename(columns={"Fine.cell.type": "cell_type"}, inplace=True)

In [43]:
fine_cell_types = metadata_joined[["slide_id", "cluster_label", "short_id", "cell_type"]]
fine_cell_types

,slide_id,cluster_label,short_id,cell_type
0,20250225-Jharna-02433-A1_Scan1.er,17,02433,Tumor
1,20250225-Jharna-02433-A1_Scan1.er,1,02433,Cycling_Tumor
2,20250225-Jharna-02433-A1_Scan1.er,17,02433,Cycling_Tumor
3,20250225-Jharna-02433-A1_Scan1.er,17,02433,Cycling_Tumor
4,20250225-Jharna-02433-A1_Scan1.er,8,02433,Th1
...,...,...,...,...
6415274,20250318-Jharna-28873-A1_Scan1.er,7,28873,Stromal_Undefined_Unknown
6415275,20250318-Jharna-28873-A1_Scan1.er,26,28873,Tumor
6415276,20250318-Jharna-28873-A1_Scan1.er,17,28873,Tumor
6415277,20250318-Jharna-28873-A1_Scan1.er,26,28873,T


In [44]:
fine_cell_types.to_csv("/gpfs/home/yb2612/yb2612_fenyo/data/Cervical_mIF/output/initial_run/celltypes/27_clusters/cell_types_fine_default.csv", index=False)